# Hybrid ALNS Runtime Evaluation: Deterministic Benchmark Protocol

---

## I. Purpose

This notebook defines a **controlled benchmark procedure** to compare:
- baseline ALNS (without ML-guided repair ranking), and
- hybrid ALNS (with offline repair model guidance).

The objective is to estimate relative performance on solution quality and computational efficiency under matched evaluation conditions.


## II. Experimental Preconditions and Reproducibility Controls

Required artifacts and assumptions:
- trained model file `repair_model.pkl`,
- benchmark instance directory (`../instances` by default),
- fixed random seed and identical iteration/time budgets across methods.

Methodological principle: comparisons are meaningful only when both solvers are evaluated on the **same instance set** under **matched resource limits**.


## III. Baseline Run (ALNS without ML Repair Ranking)

This run estimates the reference performance level of the metaheuristic when repair selection is not ML-guided. The produced CSV is the control group for all subsequent comparisons.


In [ ]:
!python solver.py \
    --instances-dir ../instances \
    --iters 2000 \
    --alns \
    --seed 42 \
    --out baseline_results.csv


## IV. Hybrid Run (ALNS with Offline Repair Model)

This run injects learned repair ranking into the neighborhood reconstruction phase via `model_path=repair_model.pkl`. All non-ML evaluation settings should remain aligned with the baseline.


In [ ]:
!python solver.py \
    --instances-dir ../instances \
    --iters 2000 \
    --alns \
    --ml-repair \
    --model-path repair_model.pkl \
    --seed 42 \
    --out hybrid_results.csv


## V. Comparative Analysis

The comparison cell computes aggregate statistics (e.g., mean bins used, deltas, and win/loss counts where applicable). Interpret improvements with attention to variance across instances, not only global averages.

Recommended reporting:
- absolute and relative gap in objective value,
- runtime distribution summary,
- per-instance dominance counts.


In [ ]:
import pandas as pd

b = pd.read_csv('baseline_results.csv')
h = pd.read_csv('hybrid_results.csv')

key = ['instance'] if 'instance' in b.columns and 'instance' in h.columns else None
if key:
    m = b.merge(h, on=key, suffixes=('_baseline', '_hybrid'))
else:
    m = pd.concat([b.add_suffix('_baseline'), h.add_suffix('_hybrid')], axis=1)

summary = {}
for col in ['bins','objective','runtime_sec']:
    cb, ch = f'{col}_baseline', f'{col}_hybrid'
    if cb in m.columns and ch in m.columns:
        summary[col] = {
            'baseline_mean': m[cb].mean(),
            'hybrid_mean': m[ch].mean(),
            'relative_change_%': 100*(m[ch].mean()-m[cb].mean())/max(abs(m[cb].mean()),1e-9),
        }

pd.DataFrame(summary).T


## VI. Interpretation Checklist

A result should be considered operationally convincing when:
1. quality gains are consistent across a broad subset of instances,
2. improvements persist across repeated seeds,
3. runtime overhead (if any) is justified by quality improvement.
